# Archive: Third-Level (Sub-)Subtopic Modeling
<!-- structured-notebook -->
## Notebook Summary
Purpose: exploratory third-level decomposition — subtopics of subtopics. Ran on a handful of the largest parent topics during early exploration.

Result: the third level was rarely worth the added complexity. Most subtopics were already specific enough that further decomposition produced either overfit micro-clusters or re-surfaced keywords that were dominated by a few posts. The published pipeline stops at subtopics (`03_topic_refinement/01_subtopic_analysis.ipynb`). This notebook is kept so a new contributor can rerun the experiment if a particular parent topic warrants deeper exploration.


In [ ]:
import webbrowser
import os
import pickle
from collections import defaultdict

import numpy as np
import pandas as pd
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans


def explore_subtopics(
    parent_topic_id: int,
    topic_id: int,
    embedding_model_name: str = "all-mpnet-base-v2",
    n_subtopics: int = 10,
    top_n_words: int = 10,
    visualize: bool = True,
) -> BERTopic:
    """
    Generate subtopics for a given parent topic, save model, topic info,
    AND subtopic->docs mapping for deeper recursion later.
    """
    save_dir = f"../../subtopic_models/topic_{parent_topic_id}_subtopics/topic_{topic_id}_subtopics"
    os.makedirs(save_dir, exist_ok=True)

    topic_model_path = f"../../subtopic_models/topic_{parent_topic_id}_subtopics/subtopic_model_topic{parent_topic_id}.model"
    topic_doc_map_path = f"../../subtopic_models/topic_{parent_topic_id}_subtopics/subtopic_doc_map_topic{parent_topic_id}.pkl"

    # Load main model and mapping
    print(f"Loading model from {topic_model_path} ...")
    topic_model = BERTopic.load(topic_model_path)

    with open(topic_doc_map_path, "rb") as f:
        topic_doc_map = pickle.load(f)

    if topic_id not in topic_doc_map or len(topic_doc_map[topic_id]) == 0:
        raise ValueError(f"No documents found for topic {topic_id}")

    docs = topic_doc_map[topic_id]
    print(f"Found {len(docs)} documents for topic {topic_id}.")

    # Cap subtopic count to available docs
    n_subtopics = max(2, min(n_subtopics, len(docs)))

    # Embed documents
    embedding_model = SentenceTransformer(embedding_model_name)
    embeddings = embedding_model.encode(docs, show_progress_bar=True)

    # Define subtopic BERTopic model
    subtopic_model = BERTopic(
        embedding_model=None,  # using precomputed embeddings
        umap_model=PCA(n_components=25),
        hdbscan_model=MiniBatchKMeans(n_clusters=n_subtopics, random_state=42),
        vectorizer_model=OnlineCountVectorizer(
            stop_words="english",
            min_df=2,
            max_df=0.9,
            ngram_range=(1, 2),
            max_features=5000
        ),
        calculate_probabilities=True,
        verbose=True
    )

    # Fit on subtopic documents
    subtopic_model.fit(docs, embeddings=embeddings)

    # ── NEW: build and save subtopic → docs mapping (and a long-form CSV)
    subtopics, probs = subtopic_model.transform(docs, embeddings=embeddings)
    max_probs = probs.max(axis=1) if probs is not None else np.full(len(docs), np.nan)

    subtopic_doc_map = defaultdict(list)
    for st_id, doc in zip(subtopics, docs):
        subtopic_doc_map[int(st_id)].append(doc)

    # Save mapping as .pkl
    map_path = os.path.join(save_dir, f"subtopic_doc_map_topic{topic_id}.pkl")
    with open(map_path, "wb") as f:
        pickle.dump(subtopic_doc_map, f)
    print(f"Saved subtopic→docs mapping to {map_path}")

    # Also save a long-form CSV (parent_topic, subtopic, doc, max_prob)
    df_assign = pd.DataFrame({
        "parent_topic": topic_id,
        "subtopic": np.array(subtopics, dtype=int),
        "doc": docs,
        "max_prob": max_probs
    })
    assign_csv = os.path.join(save_dir, f"subtopic_assignments_topic{topic_id}.csv")
    df_assign.to_csv(assign_csv, index=False)
    print(f"Saved subtopic assignments to {assign_csv}")

    # Topic info CSV
    subtopic_info_df = subtopic_model.get_topic_info()
    subtopic_info_df["TopWords"] = subtopic_info_df["Topic"].apply(
        lambda t: ", ".join([w for w, _ in (subtopic_model.get_topic(t) or [])]) if t != -1 else ""
    )

    csv_path = os.path.join(save_dir, f"subtopic_info_topic{topic_id}.csv")
    subtopic_info_df.to_csv(csv_path, index=False)
    print(f"Saved subtopic info to {csv_path}")

    # Save the subtopic model
    save_path = os.path.join(save_dir, f"subtopic_model_topic{topic_id}.model")
    subtopic_model.save(save_path)
    print(f"Saved subtopic model to {save_path}")

    # Optional visualization
    if visualize:
        fig_map = subtopic_model.visualize_topics()
        map_path = os.path.join(save_dir, "topic_map.html")
        fig_map.write_html(map_path)
        webbrowser.open("file://" + os.path.abspath(map_path))

        # Correct argument is n_words
        fig_bar = subtopic_model.visualize_barchart(n_words=top_n_words)
        bar_path = os.path.join(save_dir, "subtopic_barchart.html")
        fig_bar.write_html(bar_path)
        webbrowser.open("file://" + os.path.abspath(bar_path))

    return subtopic_model

sub_model = explore_subtopics(
    parent_topic_id=165,
    topic_id=4,
    n_subtopics=10,
    visualize=True
)
